# UATD → YOLO training (Google Colab)

Trains **YOLOv8** on the [UATD sonar dataset](https://doi.org/10.6084/m9.figshare.21331143) (VOC XML → YOLO), then downloads `best.pt` for your SEABED app.

**Before you run:** Runtime → Change runtime type → **T4 GPU** → Save.

Run cells top to bottom. Total time: ~30–90 min (download + train).

In [ ]:
# Cell 1 — install + GPU check
!pip install -q ultralytics opencv-python-headless

import torch
assert torch.cuda.is_available(), "No GPU — enable Runtime → Change runtime type → T4 GPU"
print("GPU ready:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — download UATD from figshare (Training + Test_1 for val)
import json
import urllib.request
import zipfile
from pathlib import Path

ARTICLE_ID = 21331143
RAW = Path("/content/uatd_raw")
RAW.mkdir(exist_ok=True)

meta = json.loads(urllib.request.urlopen(
    f"https://api.figshare.com/v2/articles/{ARTICLE_ID}", timeout=120
).read().decode())

WANT = {"UATD_Training.zip", "UATD_Test_1.zip"}
roots = {}
for f in meta["files"]:
    if f["name"] not in WANT:
        continue
    dest = RAW / f["name"]
    if not dest.exists():
        print("Downloading", f["name"], "...")
        urllib.request.urlretrieve(f["download_url"], dest)
    else:
        print("Already have", f["name"])
    out_dir = RAW / f["name"].replace(".zip", "")
    if not out_dir.exists():
        print("Extracting", f["name"], "...")
        with zipfile.ZipFile(dest) as z:
            z.extractall(out_dir)
    roots[f["name"]] = out_dir

TRAIN_ROOT = roots["UATD_Training.zip"]
VAL_ROOT = roots["UATD_Test_1.zip"]
print("Train root:", TRAIN_ROOT)
print("Val root:", VAL_ROOT)

In [ ]:
# Cell 3 — VOC → YOLO converter (same logic as convert_uatd.py)
import shutil
import xml.etree.ElementTree as ET
import cv2

def voc_box_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
    w = max(xmin, xmax) - min(xmin, xmax)
    h = max(ymin, ymax) - min(ymin, ymax)
    cx = min(xmin, xmax) + w / 2
    cy = min(ymin, ymax) + h / 2
    return cx / img_w, cy / img_h, w / img_w, h / img_h

def parse_voc(path):
    root = ET.parse(path).getroot()
    out = []
    for obj in root.findall("object"):
        name = (obj.findtext("name") or "").strip()
        bb = obj.find("bndbox")
        if not name or bb is None:
            continue
        out.append((name, float(bb.findtext("xmin", 0)), float(bb.findtext("ymin", 0)),
                      float(bb.findtext("xmax", 0)), float(bb.findtext("ymax", 0))))
    return out

def split_dirs(root):
    img_dir = ann_dir = None
    for p in root.iterdir():
        if not p.is_dir():
            continue
        low = p.name.lower()
        if low in ("image", "images"):
            img_dir = p
        elif low in ("annotation", "annotations"):
            ann_dir = p
    if not img_dir or not ann_dir:
        raise FileNotFoundError(f"Need image/ + annotation/ under {root}")
    return img_dir, ann_dir

def discover_classes(ann_dirs):
    names = set()
    for d in ann_dirs:
        for xml in d.glob("*.xml"):
            for name, *_ in parse_voc(xml):
                names.add(name)
    return sorted(names)

def convert_split(img_dir, ann_dir, out_images, out_labels, class_map):
    n_img = n_box = 0
    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in {".bmp", ".jpg", ".jpeg", ".png"}:
            continue
        ann_path = ann_dir / (img_path.stem + ".xml")
        if not ann_path.exists():
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        lines = []
        for name, xmin, ymin, xmax, ymax in parse_voc(ann_path):
            if name not in class_map:
                continue
            cid = class_map[name]
            cx, cy, bw, bh = voc_box_to_yolo(xmin, ymin, xmax, ymax, w, h)
            lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        out_images.mkdir(parents=True, exist_ok=True)
        out_labels.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_images / f"{img_path.stem}.jpg"), img)
        (out_labels / f"{img_path.stem}.txt").write_text(
            "\n".join(lines) + ("\n" if lines else ""))
        n_img += 1
        n_box += len(lines)
    return n_img, n_box

print("Converter loaded.")

In [ ]:
# Cell 4 — run conversion + write uatd.yaml
OUT = Path("/content/data/uatd")
if OUT.exists():
    shutil.rmtree(OUT)

train_img, train_ann = split_dirs(TRAIN_ROOT)
val_img, val_ann = split_dirs(VAL_ROOT)
CLASSES = discover_classes([train_ann, val_ann])
class_map = {c: i for i, c in enumerate(CLASSES)}
print("Classes:", CLASSES)

tr_n, tr_b = convert_split(train_img, train_ann, OUT / "images/train", OUT / "labels/train", class_map)
va_n, va_b = convert_split(val_img, val_ann, OUT / "images/val", OUT / "labels/val", class_map)
print(f"train: {tr_n} images, {tr_b} boxes")
print(f"val:   {va_n} images, {va_b} boxes")

yaml_text = "path: /content/data/uatd\ntrain: images/train\nval: images/val\n\nnames:\n"
yaml_text += "\n".join(f"  {i}: {c}" for i, c in enumerate(CLASSES))
Path("/content/uatd.yaml").write_text(yaml_text)
print("Wrote /content/uatd.yaml")

In [ ]:
# Cell 5 — train on GPU (~1–1.5 h for 50 epochs; set EPOCHS=30 for faster run)
from ultralytics import YOLO

EPOCHS = 50   # lower to 30 if you want a quicker pass
BATCH = 16

model = YOLO("yolov8n.pt")
model.train(
    data="/content/uatd.yaml",
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=0,
    name="uatd_yolo",
    project="/content/runs/detect",
)
metrics = model.val()
print("mAP50-95:", metrics.box.map, " mAP50:", metrics.box.map50)

In [ ]:
# Cell 6 — download best.pt to your computer
from google.colab import files

BEST = Path("/content/runs/detect/uatd_yolo/weights/best.pt")
assert BEST.exists(), f"Missing {BEST} — did training finish?"
print("Downloading", BEST, f"({BEST.stat().st_size / 1e6:.1f} MB)")
files.download(str(BEST))

## Back on your laptop

1. Copy the downloaded `best.pt` to:
   `runs/detect/sonar_yolo/weights/best.pt` (replace the synthetic model)
2. Restart the app: `python server.py`
3. Open http://127.0.0.1:8000 and upload a UATD sonar image

Contacts will show real UATD classes (cube, cylinder, tyre, …).

If **Download** or **Convert** errors, paste the message — figshare layout or XML tags may need a small patch.